In [ ]:
# %% [1] — Imports
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import importlib

import solenoid_lib
importlib.reload(solenoid_lib)

from solenoid_lib import (
    solenoid_length,
    theta_solenoid,
    w_tape_mm,
    t_tape_mm,
    mu0,
    Ic_tape,
    Je_tape,
    solenoid_field_center,
    solenoid_field_profile,
    magnetic_energy,
    hoop_stress,
    solenoid_summary,
)

# ─────────────────────────────────────────────────────────────────────────────
# Parameters
# ─────────────────────────────────────────────────────────────────────────────
je_fixed    = 15.31733963        # [A/mm²]  fixed operating current density
je_fixed_si = je_fixed * 1e6  # [A/m²]   SI version for solenoid functions

ri = 1.5 # [m] inner radius of solenoid
rf = 1.780241812  # [m] outer radius of solenoid
solenoid_length = 9  # [m] length of solenoid

b0 = solenoid_field_center(ri, rf, je_fixed_si, solenoid_length)

sigma_pa, b = hoop_stress(ri, rf, je_fixed_si, solenoid_length)


Ic = Ic_tape(b0, theta_solenoid)   # [A/mm²]

j_crit_mm2,je_max_mm2 = Je_tape(b0, theta_solenoid)   # [A/mm²]


print(f"  B0 at je_fiexd           = {b0:.0f} T")

print(f"  B           = {b:.0f} T")
print(f"  Ic           = {Ic:.0f} A")

print(f"  hoop stress @ B0           = {sigma_pa/1e6:.0f} MPa")
#print(f"  magnetic energy @ B0           = {E_mag/1e6:.0f} MJ")
print(f"  Jc @ B0           = {j_crit_mm2:.0f} A")
print(f"  Je @ B0           = {je_max_mm2:.0f} A")

In [ ]:
"""
Solenoid summary: peak conductor field, central field, and hoop stress
by thin-shell magnetic pressure and by Wilson thick wall.
"""

import math

MU0 = 4.0e-7 * math.pi
P0 = {0: 1.0, 2: -1/2, 4: 3/8, 6: -5/16, 8: 35/128, 10: -63/256}   # P_n(0)


def _br(g, a, b):                      # [ ... ]_{r=1}^{r=alpha}
    return g(a, b) - g(1.0, b)

def F(a, b):
    return b * math.log((a + math.hypot(a, b)) / (1.0 + math.hypot(1.0, b)))

def FE2(a, b):
    g = lambda r, b: r**3 / (r*r + b*b)**1.5
    return -_br(g, a, b) / (2 * b)

def FE4(a, b):
    g = lambda r, b: r**3*(2*r**4 + 7*r**2*b**2 + 20*b**4) / (r*r + b*b)**3.5
    return -_br(g, a, b) / (24 * b**3)

def FE6(a, b):
    g = lambda r, b: r**3*(8*r**8 + 44*r**6*b**2 + 99*r**4*b**4
                           + 28*r**2*b**6 + 280*b**8) / (r*r + b*b)**5.5
    return -_br(g, a, b) / (240 * b**5)

def FE8(a, b):
    g = lambda r, b: r**3*(16*r**12 + 120*r**10*b**2 + 390*r**8*b**4 + 715*r**6*b**6
                           + 1080*r**4*b**8 - 1008*r**2*b**10
                           + 1344*b**12) / (r*r + b*b)**7.5
    return -_br(g, a, b) / (896 * b**7)

def FE10(a, b):
    g = lambda r, b: r**3*(128*r**16 + 1216*r**14*b**2 + 5168*r**12*b**4
                           + 12920*r**10*b**6 + 20995*r**8*b**8 + 19976*r**6*b**10
                           + 49632*r**4*b**12 - 46464*r**2*b**14
                           + 21120*b**16) / (r*r + b*b)**9.5
    return -_br(g, a, b) / (11520 * b**9)

TERMS = {0: F, 2: FE2, 4: FE4, 6: FE6, 8: FE8, 10: FE10}


# ---------------------------------------------------------------- fields
def b_peak(length, r_inner, r_outer, j, nmax=10):
    """Peak conductor field at (r,z) = (a1,0), Legendre expansion at xi = 1. [T]"""
    a1 = r_inner
    alpha, beta = r_outer / a1, 0.5 * length / a1
    return MU0 * j * a1 * sum(P0[n] * TERMS[n](alpha, beta)
                              for n in sorted(TERMS) if n <= nmax)


def b_center(length, r_inner, r_outer, j):
    """On-axis central field, exact Biot-Savart over the winding cross-section. [T]"""
    zp, zm = 0.5 * length, -0.5 * length

    def log_term(z):
        return z * math.log((math.hypot(r_outer, z) + r_outer) /
                            (math.hypot(r_inner, z) + r_inner))

    return 0.5 * MU0 * j * (log_term(zp) - log_term(zm))


# ---------------------------------------------------------------- stresses
def wilson_peak(length, r_inner, r_outer, j, B1, nu=0.3, kappa=0.0, npts=201):
    """Peak Wilson hoop stress [Pa] at the bore, for a given B1."""
    a1, alpha = r_inner, r_outer / r_inner
    S = j * B1 * a1 / (alpha - 1.0)

    kA = (2 + nu) / 3 * (alpha - kappa)
    kB = (3 + nu) / 8 * (1 - kappa)

    C0 = kA * (alpha**2 + alpha + 1) / (alpha + 1) - kB * (alpha**2 + 1)
    C2 = alpha**2 * (kA / (alpha + 1) - kB)
    C1 = -(1 + 2*nu) / 3 * (alpha - kappa)
    C3 = (1 + 3*nu) / 8 * (1 - kappa)

    return max(S * (C0 + C2 / p**2 + C1 * p + C3 * p**2)
               for p in (1.0 + (alpha - 1.0) * i / (npts - 1) for i in range(npts)))


def summary(length, r_inner, r_outer, j, nu=0.3, kappa=0.0):
    """
    B1     peak conductor field at (a1,0), Legendre expansion   [T]
    B0     central field on axis, Biot-Savart                   [T]
    sig_p  thin-shell magnetic pressure (B0^2/2mu0)(a1/th)      [Pa]
    sig_w  Wilson peak hoop at the bore, using B1               [Pa]
    """
    th = r_outer - r_inner
    B1 = b_peak(length, r_inner, r_outer, j)
    B0 = b_center(length, r_inner, r_outer, j)
    sig_p = (B0**2 / (2.0 * MU0)) * (r_inner / th)
    sig_w = wilson_peak(length, r_inner, r_outer, j, B1, nu, kappa)
    je_ref    = 1e8          # [A/m²]  arbitrary reference (100 A/mm²)
    sigma_ref =750e6       # [Pa]    arbitrary reference (750 MPa)
    # σ ∝ Je²  →  Je_lim = Je_ref * sqrt(sigma_limit / sigma_ref)
    je_lim = je_ref * np.sqrt(sig_p / sigma_ref)


    print(f"\nL = {length*1e3:.1f} mm   a1 = {r_inner*1e3:.1f} mm   "
          f"a2 = {r_outer*1e3:.1f} mm   j = {j/1e6:.1f} A/mm^2")
    print(f"B1  (peak, Legendre)         = {B1:8.4f} T")
    print(f"B0  (centre, Biot-Savart)    = {B0:8.4f} T")
    print(f"sigma_hoop (mag. pressure)   = {sig_p/1e6:8.2f} MPa")
    print(f"sigma_hoop (Wilson, peak)    = {sig_w/1e6:8.2f} MPa")
    print(f"Je_lim (σ_ref = 750 MPa)     = {je_lim/1e6:8.2f} A/mm^2")
    return B1, B0, sig_p, sig_w


summary(length=9, r_inner=1.5, r_outer=1.780241812, j=15.31733963e6)

In [ ]:
import numpy as np
import solenoid_lib


ri          = 1.5                # inner radius [m]
th          = 0.280241812                 # radial build [m]
L_req       = 9.0                        # requested length [m]
theta       = solenoid_lib.theta_solenoid
sigma_limit = 750e6                      # hoop limit [Pa]
max_iter    = 50
mu0         = 4.0e-7 * np.pi

t_tape = solenoid_lib.t_tape_mm * 1e-3   # tape thickness [m]
w_tape = solenoid_lib.w_tape_mm * 1e-3   # tape width [m]
a_tape = t_tape * w_tape                 # one turn in the r-z plane [m2]

# Axial pitch is the tape width, so the length quantises exactly;
# the (1 - f_tape) dilution sits radially, so Th does not.
n_pc = max(1, round(L_req / w_tape))
L    = n_pc * w_tape
rf   = ri + th

# ── self-consistent Je: stress limit vs tape limit at its own field ──────────
je_mm2    = solenoid_lib.je_max_stress_limited(ri, rf, L, sigma_limit) / 1e6
limit     = "mechanical"
converged = False

print(f"{'it':>3} {'Je_pack':>10} {'B0':>8} {'Je_tape':>10} {'f_tape':>9}")
print("-" * 44)

for k in range(1, max_iter + 1):
    b0 = solenoid_lib.solenoid_field_center(ri, rf, je_mm2 * 1e6, L)
    _, je_tape_mm2 = solenoid_lib.Je_tape(b0, theta)

    f_tape = je_mm2 / je_tape_mm2                # tape fraction the pack needs
    print(f"{k:3d} {je_mm2:10.2f} {b0:8.3f} {je_tape_mm2:10.2f} {f_tape:9.4f}")

    if f_tape <= 1.0:
        converged = True
        break

    je_mm2 = je_tape_mm2                         # EM limited: drop to tape limit
    limit  = "EM"

# Do not clamp f_tape to 1: that would hide an infeasible point as a valid one.
if not converged:
    raise RuntimeError(f"no self-consistent Je after {max_iter} iterations "
                       f"(f_tape = {f_tape:.4f} > 1)")

# ── re-evaluate everything at the final Je so the table is self-consistent ───
je_si          = je_mm2 * 1e6
b0             = solenoid_lib.solenoid_field_center(ri, rf, je_si, L)
sigma_pa, _    = solenoid_lib.hoop_stress(ri, rf, je_si, L)
_, je_tape_mm2 = solenoid_lib.Je_tape(b0, theta)
f_tape         = je_mm2 / je_tape_mm2

# ── winding layout: integer turns, rounded up so the pack is never short ─────
n_tp     = int(np.ceil(f_tape * th / t_tape))
n_tot    = n_tp * n_pc
a_wind   = th * L                                # current-carrying cross-section
f_built  = n_tot * a_tape / a_wind               # as-built tape fraction
i_turn   = je_si * a_wind / n_tot                # current per turn [A]
i_max    = je_tape_mm2 * solenoid_lib.t_tape_mm * solenoid_lib.w_tape_mm
len_tape = n_tot * np.pi * (ri + rf)

pitch  = th / n_tp                               # radial space per turn [m]
a_cell = pitch * w_tape                          # cross-section one turn owns [m2]

print(f"\n{'quantity':<18}{'unit':<9}{'value':>12}")
print("-" * 39)
for label, unit, val, fmt in [
    ("Ri",              "mm",      ri * 1e3,             "12.1f"),
    ("Rf",              "mm",      rf * 1e3,             "12.1f"),
    ("Th",              "mm",      th * 1e3,             "12.3f"),
    ("Magnet length",   "mm",      L * 1e3,              "12.3f"),
    ("radial pitch",    "mm",      pitch * 1e3,          "12.4f"),
    ("turns / pancake", "-",       n_tp,                 "12d"),
    ("pancakes",        "-",       n_pc,                 "12d"),
    ("limited by",      "-",       limit,                ">12s"),
    ("Je_pack",         "A/mm2",   je_mm2,               "12.2f"),
    ("Je_tape @B0",     "A/mm2",   je_tape_mm2,          "12.2f"),
    ("J_tape built",    "A/mm2",   je_mm2 / f_built,     "12.2f"),
    ("f_tape min",      "%",       f_tape * 100,         "12.2f"),
    ("f_tape built",    "%",       f_built * 100,        "12.2f"),
    ("B0",              "T",       b0,                   "12.3f"),
    ("sigma_hoop",      "MPa",     sigma_pa / 1e6,       "12.1f"),
    ("sigma limit",     "MPa",     sigma_limit / 1e6,    "12.1f"),
    ("I_turn",          "A",       i_turn,               "12.1f"),
    ("I_tape max @B0",  "A",       i_max,                "12.1f"),
    ("margin",          "-",       i_max / i_turn,       "12.3f"),
    ("NI",              "MA-turn", n_tot * i_turn / 1e6, "12.3f"),
    ("L_tape",          "km",      len_tape / 1e3,       "12.2f"),
]:
    print(f"{label:<18}{unit:<9}{val:{fmt}}")
print("-" * 39)

# ═════════════════════════════════════════════════════════════════════
# QUENCH PROTECTION — adiabatic hot-spot check
# ═════════════════════════════════════════════════════════════════════

fCu       = 0.50      # copper fraction of the tape cross-section [-]
R_EE      = 4.0       # dump resistance [ohm]
t_det     = 0.100     # detection + validation delay [s]  (not in ql_tot below)
gamma_max = 5.0e16 * fCu   # copper action limit at the hot spot [A2 s m-4]


# ── operating current ────────────────────────────────────────────────
i0 = i_turn                            # current in one tape [A]

# ── inductance from the stored magnetic energy ───────────────────────
# je_si already carries the f_tape dilution, so the correct current
# distribution is the real winding rectangle (ri -> rf) at je_si.
em_tot = solenoid_lib.magnetic_energy(ri, rf, je_si, L)      # whole magnet [J]
em_iso = solenoid_lib.magnetic_energy(ri, rf, je_si, w_tape) # one pancake [J]

l_tot = 2.0 * em_tot / i0 ** 2         # series-equivalent magnet [H]
l_iso = 2.0 * em_iso / i0 ** 2         # isolated pancake, self only [H]

# ── dump topology ────────────────────────────────────────────────────
# "isolated": one resistor across the pancake being dumped. The other
#   n_pc - 1 pancakes are held at flat-top by their own supplies, so
#   dI_j/dt = 0 for j != i and no mutual term enters the loop equation.
#   The dump sees the pancake self-inductance only. Costs 2*n_pc leads.
# "series": pancakes in series, one resistor for the whole magnet. Same
#   current per turn, but the dump sees l_tot and the terminal voltage
#   is R_EE*i0 across the full stack instead of across one pancake.
#topology = "isolated"
topology = "series"

l_dump = l_iso if topology == "isolated" else l_tot

# ── dump circuit ─────────────────────────────────────────────────────
tau_ee = l_dump / R_EE                 # decay time constant [s]
u_ee   = R_EE * i0                     # dump voltage [V]

# ── copper ───────────────────────────────────────────────────────────
s_cu = a_tape * fCu                    # copper area per turn [m2]
j_cu = i0 / s_cu                       # copper current density [A/m2]

# ── MIITs (current action) ───────────────────────────────────────────
# Dump term only. The t_det plateau (i0**2 * t_det) is NOT included here.
# Note 0.5*i0**2*tau = E_dump/R_EE exactly, so i0 cancels: the quench load
# depends only on the extracted energy and the resistor, not on the current.
ql_tot = 0.5 * i0 ** 2 * tau_ee        # quench load [A2 s]

gamma_alternative = ql_tot / s_cu ** 2     # same number, written the other way
gamma             = j_cu ** 2 * tau_ee / 2
margin_gamma      = gamma - gamma_max      # must be negative

# ── copper needed ────────────────────────────────────────────────────
cu_needed_total = np.sqrt(ql_tot / gamma_max)          # [m2] Cu per turn needed
a_cu_needed     = max(cu_needed_total - s_cu, 0.0)     # [m2] Cu per turn to add
th_cu_needed    = a_cu_needed / w_tape                 # [m]  thickness per turn

# A turn owns the radial pitch, not the whole build: dividing by th would
# understate the fraction by n_tp. Smeared equivalent: f_Cu = Je/J_Cu,max.
f_cu_req         = cu_needed_total /w_tape / th            # total Cu fraction [-]
f_cu_have        = s_cu/w_tape / th                       # already in the tape [-]
f_cu_add         = max(f_cu_req - f_cu_have, 0.0)      # to be added [-]
th_cu_percentage = f_cu_add * 100                      # [%] of the build to add

th_cu_req = f_cu_req * th              # [m] Cu summed over the build
th_cu_add = f_cu_add * th              # [m] Cu to add over the build

print(f"\n{'quench':<18}{'unit':<9}{'value':>12}")
print("-" * 39)
for label, unit, val, fmt in [
    ("I0",                "A",       i0,                   "12.1f"),
    ("E magnetic",        "MJ",      em_tot / 1e6,         "12.2f"),
    ("E dumped",          "MJ",      0.5 * l_dump * i0 ** 2 / 1e6, "12.2f"),
    ("L ",                "H",       l_dump,               "12.4f"),
    ("R_EE",              "ohm",     R_EE,                 "12.2f"),
    ("U_EE",              "V",       u_ee,                 "12.1f"),
    ("tau_EE",            "ms",      tau_ee * 1e3,         "12.3f"),
    ("S_Cu",              "mm2",     s_cu * 1e6,           "12.4f"),
    ("J_Cu",              "A/mm2",   j_cu / 1e6,           "12.1f"),
    ("MIITs ",            "MIIt",    ql_tot / 1e6,         "12.5f"),
    ("gamma",             "A2s/m4",  gamma,                "12.3e"),
    ("gamma alternative", "A2s/m4",  gamma_alternative,    "12.3e"),
    ("gamma max",         "A2s/m4",  gamma_max,            "12.3e"),
    ("margin",            "A2s/m4",  margin_gamma,         "12.3e"),
    ("S_Cu required",     "mm2",     cu_needed_total * 1e6,"12.4f"),
    ("Cu needed / turn",  "mm2",     a_cu_needed * 1e6,    "12.4f"),
    ("t_Cu / turn",       "mm",      th_cu_needed * 1e3,   "12.4f"),
    ("radial pitch",      "mm",      pitch * 1e3,          "12.4f"),
]:
    print(f"{label:<18}{unit:<9}{val:{fmt}}")
print("-" * 39)

# ═════════════════════════════════════════════════════════════════════
# MECHANICAL RE-CHECK — copper is the only non-structural material
# ═════════════════════════════════════════════════════════════════════

# Ri, Th and L are fixed, so copper changes neither the Lorentz load nor
# sigma_hoop. It changes what is left to carry it: the structural material
# is th*(1 - f_Cu), so the real stress there is sigma_hoop/(1 - f_Cu).
# Tape and Hastelloy filler are the same material and both count as structure.
f_struct     = 1.0 - f_cu_req
sigma_struct = sigma_pa / f_struct if f_struct > 0.0 else np.inf
util         = sigma_struct / sigma_limit

print(f"\n{'copper / mechanical':<22}{'unit':<9}{'value':>12}")
print("-" * 43)
for label, unit, val, fmt in [
    ("Th",              "mm",   th * 1e3,             "12.1f"),
    ("f_tape built",    "%",    f_built * 100,        "12.2f"),
    ("f_Cu in tape",    "%",    f_cu_have * 100,      "12.2f"),
    ("f_Cu required",   "%",    f_cu_req * 100,       "12.2f"),
    ("f_Cu to add",     "%",    f_cu_add * 100,       "12.2f"),
    ("Th Cu required",  "mm",   th_cu_req * 1e3,      "12.1f"),
    ("Th Cu to add",    "mm",   th_cu_add * 1e3,      "12.1f"),
    ("f_structural",    "%",    f_struct * 100,       "12.2f"),
    ("sigma_hoop",      "MPa",  sigma_pa / 1e6,       "12.1f"),
    ("sigma structure", "MPa",  sigma_struct / 1e6,   "12.1f"),
    ("sigma limit",     "MPa",  sigma_limit / 1e6,    "12.1f"),
    ("utilisation",     "-",    util,                 "12.3f"),
]:
    print(f"{label:<22}{unit:<9}{val:{fmt}}")

if f_cu_req >= 1.0:
    print(f"{'verdict':<22}{'-':<9}{'IMPOSSIBLE':>12}")
    print(f"  needs {f_cu_req * 100:.0f} % copper, more than the whole build")
elif f_built + f_cu_add > 1.0:
    print(f"{'verdict':<22}{'-':<9}{'NO FIT':>12}")
    print(f"  tape {f_built * 100:.1f} % + copper {f_cu_add * 100:.1f} % "
          f"= {(f_built + f_cu_add) * 100:.1f} % of the build")
elif util > 1.0:
    print(f"{'verdict':<22}{'-':<9}{'FAIL':>12}")
    print(f"  structure over by {(util - 1.0) * 100:.1f} % -> lower Je")
else:
    print(f"{'verdict':<22}{'-':<9}{'PASS':>12}")
    print(f"  {(1.0 - util) * 100:.1f} % structural margin, "
          f"{(1.0 - f_built - f_cu_add) * 100:.1f} % free for Hastelloy")
print("-" * 43)

In [ ]:
import numpy as np
import solenoid_lib

# ═════════════════════════════════════════════════════════════════════
# INPUTS
# ═════════════════════════════════════════════════════════════════════

ri          = 0.2767070707070705                # inner radius [m]
th          = 0.08492011               # radial build [m]
L_req       = 6                        # requested length [m]

sigma_limit = 750e6         # hoop limit of the structure [Pa]
fCu         = 0.50          # copper fraction of the tape itself [-]
gamma_cu    = 5.0e16        # copper action limit at the hot spot [A2 s m-4]
sf_quench   = 1.00          # safety factor on required Cu area

topology  = "series"        # "isolated" | "series"
dump_mode = "resistance"    # "resistance"
R_EE_set  = 4.0             # [ohm] used when dump_mode == "resistance"


u_target    = 0.995         # structural utilisation to sit at [-]
fill_target = 0.98          # tape + added copper packing ceiling [-]


theta     = solenoid_lib.theta_solenoid
gamma_max = gamma_cu * fCu

# ── fixed geometry: axial pitch is the tape width, so L quantises exactly ────
t_tape = solenoid_lib.t_tape_mm * 1e-3
w_tape = solenoid_lib.w_tape_mm * 1e-3
a_tape = t_tape * w_tape
n_pc   = max(1, round(L_req / w_tape))
L      = n_pc * w_tape
rf     = ri + th
a_wind = th * L

# ── scan time ────────────────────────────────────────────────────────────────
v_bore = np.pi * ri ** 2 * L

def scan_time(b0):
    return solenoid_lib.scan_time(b0, v_bore)

def evaluate(je_mm2):

    d  = {"je": je_mm2}
    je = je_mm2 * 1e6

    # ── EM ───────────────────────────────────────────────────────────
    b0          = solenoid_lib.solenoid_field_center(ri, rf, je, L)

    sigma_pa, _ = solenoid_lib.hoop_stress(ri, rf, je, L)
    _, je_tape_mm2 = solenoid_lib.Je_tape(b0, theta)
    d.update(b0=b0, sigma_pa=sigma_pa, je_tape=je_tape_mm2,
             scan=scan_time(b0))

    if not np.isfinite(je_tape_mm2) or je_tape_mm2 <= 0.0:
        d.update(r=np.inf, binding="EM", feasible=False)
        return d                                   # tape carries nothing here

    # ── winding layout: integer turns, rounded up, pack is never short ──
    f_tape  = je_mm2 / je_tape_mm2
    n_tp    = max(1, int(np.ceil(f_tape * th / t_tape)))
    n_tot   = n_tp * n_pc
    f_built = n_tot * a_tape / a_wind
    pitch   = th / n_tp
    i0      = je * a_wind / n_tot                  # current per tape [A]
    i_max   = je_tape_mm2 * solenoid_lib.t_tape_mm * solenoid_lib.w_tape_mm
    d.update(f_tape=f_tape, n_tp=n_tp, n_tot=n_tot, f_built=f_built,
             pitch=pitch, i0=i0, i_max=i_max,
             len_tape=n_tot * np.pi * (ri + rf))

    # ── inductance from the stored energy; je already carries f_tape ──
    em_tot = solenoid_lib.magnetic_energy(ri, rf, je, L)
    em_iso = solenoid_lib.magnetic_energy(ri, rf, je, w_tape)
    l_tot  = 2.0 * em_tot / i0 ** 2
    l_iso  = 2.0 * em_iso / i0 ** 2

    # "isolated": one resistor per pancake, the dump sees the pancake self-inductance only
    # "series": one resistor for the stack, dump sees l_tot.
    l_dump  = l_iso if topology == "isolated" else l_tot
    em_dump = em_iso if topology == "isolated" else em_tot

    # ── dump circuit ─────────────────────────────────────────────────
    r_ee   = R_EE_set
    tau_ee = l_dump / r_ee
    u_ee   = r_ee * i0
    d.update(em_tot=em_tot, em_iso=em_iso, em_dump=em_dump,
             l_tot=l_tot, l_iso=l_iso, l_dump=l_dump,
             r_ee=r_ee, tau_ee=tau_ee, u_ee=u_ee)

    # ── hot spot, adiabatic: exponential dump only ───────────────────
    # per turn  ql = i0^2 * tau/2,  and i0^2*tau/2 = E/R.
    # Smeared over the build j_cu = Je/f_cu, so n_tp cancels here.
    ql_dump = 0.5 * i0 ** 2 * tau_ee        # quench load [A2 s]

    j_cu_max  = np.sqrt(gamma_max / (0.5 * tau_ee))
    f_cu_req  = sf_quench * je / j_cu_max          # total Cu fraction of build
    f_cu_have = f_built * fCu                      # Cu already inside the tape
    f_cu_add  = max(f_cu_req - f_cu_have, 0.0)     # co-wound Cu to add
    f_cu_eff  = max(f_cu_req, f_cu_have)           # Cu actually in the build

    s_cu = f_cu_eff * a_wind / n_tot               # copper area per turn [m2]

    d.update(ql_dump=ql_dump, j_cu_max=j_cu_max, f_cu_req=f_cu_req,
             f_cu_have=f_cu_have, f_cu_add=f_cu_add, f_cu_eff=f_cu_eff,
             s_cu=s_cu, j_cu=je / f_cu_eff,
             gamma_op=(je / f_cu_eff) ** 2 * (0.5 * tau_ee))

    assert d["gamma_op"] <= gamma_max * (1 + 1e-9), "hot-spot sizing inconsistent"

    # ── mechanics: copper carries nothing, tape and filler are the same ──
    # Ri, Th, L are fixed, so copper changes neither the Lorentz load nor
    # sigma_hoop. It changes what is left to carry it: th*(1 - f_Cu).
    f_struct     = 1.0 - f_cu_eff
    sigma_struct = sigma_pa / f_struct if f_struct > 0.0 else np.inf
    util         = sigma_struct / sigma_limit
    fill         = f_built + f_cu_add
    d.update(f_struct=f_struct, sigma_struct=sigma_struct,
             util=util, fill=fill)

    # ── binding constraint ───────────────────────────────────────────
    r_mech = util / u_target
    r_fill = fill / fill_target
    d["r"] = max(r_mech, r_fill)
    d["binding"] = ("EM/tape" if f_tape > 1.0 else
                    "mechanical" if r_mech >= r_fill else "packing")
    d["feasible"] = d["r"] <= 1.0
    return d


# ═════════════════════════════════════════════════════════════════════
# SOLVE — r(Je) is monotone increasing (jumps at each new layer go UP),
#         so the feasible set is (0, Je*] and bisection is exact.
# ═════════════════════════════════════════════════════════════════════
RTOL = 1e-12          # relative width of the final bracket

# upper bound: all of Th as structure, zero copper -> infeasible by construction
je_hi = solenoid_lib.je_max_stress_limited(ri, rf, L, sigma_limit) / 1e6
n_ev  = 0

def probe(je):
    global n_ev
    n_ev += 1
    return evaluate(je)

d_hi = probe(je_hi)
while d_hi["r"] <= 1.0:                       # only if quench costs nothing
    je_hi *= 2.0
    d_hi = probe(je_hi)

je_lo, d_lo = None, None                      # lower bound: halve until it fits
je = je_hi
for _ in range(200):
    je *= 0.5
    d = probe(je)
    if d["r"] <= 1.0:
        je_lo, d_lo = je, d
        break
    je_hi = je
if d_lo is None:
    raise RuntimeError("infeasible for every Je: check gamma_max, R_EE, fill_target")

while je_hi - je_lo > RTOL * je_lo:           # ~50 evaluations, cannot fail
    je_mid = 0.5 * (je_lo + je_hi)
    d = probe(je_mid)
    if d["r"] <= 1.0:
        je_lo, d_lo = je_mid, d
    else:
        je_hi = je_mid

best = d_lo
d    = best
print(f"converged: {n_ev} evaluations, Je* = {d['je']:.4f} A/mm2 "
      f"(bracket {je_hi - je_lo:.2e}), binding = {d['binding']}, "
      f"util = {d['util']:.4f}, fill = {d['fill']:.4f}, n_tp = {d['n_tp']}")

print(f"  budget: tape {d['f_built']*100:5.2f} %  Cu added {d['f_cu_add']*100:5.2f} % "
      f"structure {(1-d['fill'])*100:5.2f} %  |  of which Cu total "
      f"{d['f_cu_req']*100:5.2f} %, load-bearing {d['f_struct']*100:5.2f} %")

k_s   = solenoid_lib.hoop_stress(ri, rf, 1e8, L)[0] / 1e16      # sigma = k_s Je^2 [SI]
a     = np.sqrt(0.5 * d["tau_ee"] / gamma_max)
b     = k_s / (u_target * sigma_limit)
je_q  = (-a + np.sqrt(a * a + 4 * b)) / (2 * b) / 1e6           # [A/mm2]



# ═════════════════════════════════════════════════════════════════════
# FINAL MAGNET — converged design vs. the stress-limited starting guess
# ═════════════════════════════════════════════════════════════════════
rho_cu   = 8960.0
v_build  = np.pi * (rf ** 2 - ri ** 2) * L

je_seed = solenoid_lib.je_max_stress_limited(ri, rf, L, sigma_limit) / 1e6
d0      = evaluate(je_seed)

g = lambda k: d0.get(k, np.nan)

rows = [
    ("── geometry (fixed) ──", "", None, None, None),
    ("Ri",                "mm",      ri * 1e3,          ri * 1e3,          "12.1f"),
    ("Rf",                "mm",      rf * 1e3,          rf * 1e3,          "12.1f"),
    ("Th",                "mm",      th * 1e3,          th * 1e3,          "12.3f"),
    ("length",            "mm",      L * 1e3,           L * 1e3,           "12.1f"),
    ("build volume",      "m3",      v_build,           v_build,           "12.4f"),

    ("── operating point ──", "", None, None, None),
    ("Je ",           "A/mm2",   g("je"),           d["je"],           "12.2f"),
    ("B0 centre",         "T",       g("b0"),           d["b0"],           "12.3f"),
    ("scan time",         "yr",      g("scan"),         d["scan"],         "12.4g"),
    ("E stored",          "MJ",      g("em_tot") / 1e6, d["em_tot"] / 1e6, "12.2f"),
    ("L self",            "H",       g("l_tot"),        d["l_tot"],        "12.3f"),

    ("── winding ──", "", None, None, None),
    ("turns / pancake",   "-",       g("n_tp"),         d["n_tp"],         "12.0f"),
    ("pancakes",          "-",       n_pc,              n_pc,              "12.0f"),
    ("turns total",       "-",       g("n_tot"),        d["n_tot"],        "12.0f"),
    ("radial pitch",      "mm",      g("pitch") * 1e3,  d["pitch"] * 1e3,  "12.4f"),
    ("I per turn",        "A",       g("i0"),           d["i0"],           "12.1f"),
    ("I tape max @B0",    "A",       g("i_max"),        d["i_max"],        "12.1f"),
    ("margin I_max/I0",   "-",       g("i_max") / g("i0"),
                                     d["i_max"] / d["i0"],                 "12.3f"),
    ("Je tape @B0",       "A/mm2",   g("je_tape"),      d["je_tape"],      "12.2f"),
    ("J tape as built",   "A/mm2",   g("je") / g("f_built"),
                                     d["je"] / d["f_built"],               "12.2f"),
    ("f_tape needed",     "%",       g("f_tape") * 100, d["f_tape"] * 100, "12.2f"),
    ("f_tape built",      "%",       g("f_built") * 100, d["f_built"] * 100, "12.2f"),
    ("tape length",       "km",      g("len_tape") / 1e3, d["len_tape"] / 1e3, "12.2f"),

    ("── quench ──", "", None, None, None),
    ("topology",          "-",       topology,          topology,          "s"),
    ("dump mode",         "-",       dump_mode,         dump_mode,         "s"),
    ("L dump",            "H",       g("l_dump"),       d["l_dump"],       "12.4f"),
    ("E dumped",          "MJ",      g("em_dump") / 1e6, d["em_dump"] / 1e6, "12.3f"),
    ("R_EE",              "ohm",     g("r_ee"),         d["r_ee"],         "12.3f"),
    ("U_EE",              "V",       g("u_ee"),         d["u_ee"],         "12.1f"),
    ("tau_EE",            "s",       g("tau_ee"),       d["tau_ee"],       "12.4f"),
    ("J_Cu allowed",      "A/mm2",   g("j_cu_max") / 1e6, d["j_cu_max"] / 1e6, "12.1f"),
    ("f_Cu required",     "%",       g("f_cu_req") * 100, d["f_cu_req"] * 100, "12.2f"),
    ("f_Cu in tape",      "%",       g("f_cu_have") * 100, d["f_cu_have"] * 100, "12.2f"),
    ("f_Cu co-wound",     "%",       g("f_cu_add") * 100, d["f_cu_add"] * 100, "12.2f"),
    ("t_Cu co-wound",     "mm",      g("f_cu_add") * g("pitch") * 1e3,
                                     d["f_cu_add"] * d["pitch"] * 1e3,     "12.4f"),
    ("Cu mass",           "kg",      rho_cu * g("f_cu_req") * v_build,
                                     rho_cu * d["f_cu_req"] * v_build,     "12.1f"),

    ("── mechanics ──", "", None, None, None),
    ("sigma_hoop smeared", "MPa",    g("sigma_pa") / 1e6, d["sigma_pa"] / 1e6, "12.1f"),
    ("f_structural",      "%",       g("f_struct") * 100, d["f_struct"] * 100, "12.2f"),
    ("sigma structure",   "MPa",     g("sigma_struct") / 1e6,
                                     d["sigma_struct"] / 1e6,              "12.1f"),
    ("sigma limit",       "MPa",     sigma_limit / 1e6, sigma_limit / 1e6, "12.1f"),
    ("utilisation",       "-",       g("util"),         d["util"],         "12.4f"),
    ("packing",           "%",       g("fill") * 100,   d["fill"] * 100,   "12.2f"),
    ("binding",           "-",       g("binding") if d0 else "-",
                                     d["binding"],                          "s"),
]

w = 24 + 9 + 12 + 12 + 10
print(f"\n{'FINAL MAGNET':<24}{'unit':<9}{'seed':>12}{'converged':>12}{'delta %':>10}")
print("=" * w)
for label, unit, v0, v1, fmt in rows:
    if fmt is None:
        print(f"{label}")
        continue
    if fmt == "s":
        print(f"{label:<24}{unit:<9}{str(v0):>12}{str(v1):>12}{'':>10}")
        continue
    ok0 = v0 is not None and np.isfinite(v0)
    ok1 = v1 is not None and np.isfinite(v1)
    s0 = f"{v0:{fmt}}" if ok0 else f"{'-':>12}"
    s1 = f"{v1:{fmt}}" if ok1 else f"{'-':>12}"
    s2 = f"{(v1 / v0 - 1.0) * 100:10.2f}" if (ok0 and ok1 and v0 != 0.0) else f"{'-':>10}"
    print(f"{label:<24}{unit:<9}{s0}{s1}{s2}")
print("=" * w)

# ── consistency checks: these must hold if nothing is cached ────────
if d0 is not None and np.isfinite(g("je")):
    dj = d["je"] / g("je") - 1.0
    db = d["b0"] / g("b0") - 1.0
    ds = d["sigma_pa"] / g("sigma_pa") - 1.0
    print(f"  B0 tracks Je     : {db * 100:+7.3f} % vs {dj * 100:+7.3f} %  "
          f"{'ok' if abs(db - dj) < 1e-6 else 'MISMATCH'}")
    print(f"  sigma tracks Je^2: {ds * 100:+7.3f} % vs "
          f"{((1 + dj) ** 2 - 1) * 100:+7.3f} %  "
          f"{'ok' if abs(ds - ((1 + dj) ** 2 - 1)) < 1e-6 else 'MISMATCH'}")
print(f"  verdict          : "
      f"{'PASS' if (d['util'] <= 1.0 and d['fill'] <= 1.0) else 'FAIL'}, "
      f"limited by {d['binding']}, {d['je']:.2f} A/mm2 at {d['b0']:.3f} T")

RuntimeError: non-physical stored energy: em_tot=-7.191e+08 em_iso=2.536e+03 at ri=0.2767070707070705 rf=0.3616271807070705 je=1.958e+08 L=6.0